In [2]:
import os
import glob
import requests
import pandas as pd
from datetime import datetime
from trafilatura import extract
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer

In [11]:
CSV_FOLDER = r"C:\Users\User\dev\training_model\gdelt_data_fetch\gdelt_gold_filtered"  
OUTPUT_FILE = r"final_output.csv"

In [4]:
def get_article_text(url):
    try:
        r = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        if r.status_code == 200:
            text = extract(r.text)
            return text
    except Exception:
        return None
    return None

In [5]:
def get_article_text_any(url):
    text = get_article_text(url)
    if text:
        return text

    api = "https://archive.org/wayback/available"
    try:
        res = requests.get(api, params={"url": url}, timeout=10).json()
        snapshot = res.get("archived_snapshots", {}).get("closest", {})
        if snapshot and snapshot.get("available"):
            archived_url = snapshot["url"]
            text = get_article_text(archived_url)
            if text:
                return text
    except Exception:
        return None
    return None

In [6]:
def summarize_text(text, sentence_count=3):
    if not text:
        return ""
    parser = PlaintextParser.from_string(text, Tokenizer("english"))
    summarizer = LsaSummarizer()
    summary_sentences = summarizer(parser.document, sentence_count)
    return " ".join(str(s) for s in summary_sentences)

In [12]:
print(glob.glob(os.path.join(CSV_FOLDER, "*.csv")))

['C:\\Users\\User\\dev\\training_model\\gdelt_data_fetch\\gdelt_gold_filtered\\20140101_gold_filtered.csv', 'C:\\Users\\User\\dev\\training_model\\gdelt_data_fetch\\gdelt_gold_filtered\\20140102_gold_filtered.csv', 'C:\\Users\\User\\dev\\training_model\\gdelt_data_fetch\\gdelt_gold_filtered\\20140103_gold_filtered.csv', 'C:\\Users\\User\\dev\\training_model\\gdelt_data_fetch\\gdelt_gold_filtered\\20140104_gold_filtered.csv', 'C:\\Users\\User\\dev\\training_model\\gdelt_data_fetch\\gdelt_gold_filtered\\20140105_gold_filtered.csv', 'C:\\Users\\User\\dev\\training_model\\gdelt_data_fetch\\gdelt_gold_filtered\\20140106_gold_filtered.csv', 'C:\\Users\\User\\dev\\training_model\\gdelt_data_fetch\\gdelt_gold_filtered\\20140107_gold_filtered.csv', 'C:\\Users\\User\\dev\\training_model\\gdelt_data_fetch\\gdelt_gold_filtered\\20140108_gold_filtered.csv', 'C:\\Users\\User\\dev\\training_model\\gdelt_data_fetch\\gdelt_gold_filtered\\20140109_gold_filtered.csv', 'C:\\Users\\User\\dev\\training_mode

In [14]:
all_data = []

for file in glob.glob(os.path.join(CSV_FOLDER, "*.csv")):
    try:
        filename = os.path.basename(file).replace(".csv", "")
        date_str = filename[:8]
        date_obj = datetime.strptime(date_str, "%Y%m%d").date()
    except ValueError:
        print(f"Skipping file (invalid date format): {file}")
        continue

    print(f"Processing {date_obj} ...")
    df = pd.read_csv(file)

    daily_texts = []
    for url in df["SOURCEURL"]:
        if not isinstance(url, str) or not url.startswith("http"):
            continue
        article_text = get_article_text_any(url)
        if article_text:
            daily_texts.append(article_text)

    if daily_texts:
        combined_text = " ".join(daily_texts)
        summary = summarize_text(combined_text, sentence_count=3)
    else:
        summary = "No articles could be retrieved for this date."

    all_data.append({"date": date_obj, "summary": summary})

final_df = pd.DataFrame(all_data)
final_df.sort_values("date", inplace=True)
final_df.to_csv(OUTPUT_FILE, index=False)
print(f"Output saved to {OUTPUT_FILE}")

Processing 2014-01-01 ...


KeyboardInterrupt: 